In [2]:
import os
import pandas as pd

# Path to dataset
FILE_PATH = "../data/raw/reddit_wsb.csv"

# Load dataset
df = pd.read_csv(FILE_PATH)

# Display first rows
df.head()

,title,score,id,url,comms_num,created,body,timestamp
0,"It's not about the money, it's about sending a...",55,l6ulcx,https://v.redd.it/6j75regs72e61,6,1.611863e+09,NaN,2021-01-28 21:37:41
1,Math Professor Scott Steiner says the numbers ...,110,l6uibd,https://v.redd.it/ah50lyny62e61,23,1.611862e+09,NaN,2021-01-28 21:32:10
2,Exit the system,0,l6uhhn,https://www.reddit.com/r/wallstreetbets/commen...,47,1.611862e+09,The CEO of NASDAQ pushed to halt trading “to g...,2021-01-28 21:30:35
3,NEW SEC FILING FOR GME! CAN SOMEONE LESS RETAR...,29,l6ugk6,https://sec.report/Document/0001193125-21-019848/,74,1.611862e+09,NaN,2021-01-28 21:28:57
4,"Not to distract from GME, just thought our AMC...",71,l6ufgy,https://i.redd.it/4h2sukb662e61.jpg,156,1.611862e+09,NaN,2021-01-28 21:26:56


In [3]:
df.info()
df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53187 entries, 0 to 53186
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   title      53187 non-null  object 
 1   score      53187 non-null  int64  
 2   id         53187 non-null  object 
 3   url        53187 non-null  object 
 4   comms_num  53187 non-null  int64  
 5   created    53187 non-null  float64
 6   body       24738 non-null  object 
 7   timestamp  53187 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 3.2+ MB


Index(['title', 'score', 'id', 'url', 'comms_num', 'created', 'body',
       'timestamp'],
      dtype='object')

In [6]:
df = pd.read_csv("../data/raw/reddit_wsb.csv")

df['text'] = df['title'].fillna('') + ' ' + df['body'].fillna('')
df['date'] = pd.to_datetime(df['timestamp']).dt.date
df = df[['text', 'date']]
df.head()

,text,date
0,"It's not about the money, it's about sending a...",2021-01-28
1,Math Professor Scott Steiner says the numbers ...,2021-01-28
2,Exit the system The CEO of NASDAQ pushed to ha...,2021-01-28
3,NEW SEC FILING FOR GME! CAN SOMEONE LESS RETAR...,2021-01-28
4,"Not to distract from GME, just thought our AMC...",2021-01-28


In [37]:
import re
from collections import Counter

# Pattern: 1–5 uppercase letters (possible ticker)
ticker_pattern = re.compile(r'\b[A-Z]{1,5}\b')

def extract_caps(text):
    return ticker_pattern.findall(str(text))

df['caps_tokens'] = df['text'].apply(extract_caps)

all_tokens = [t for lst in df['caps_tokens'] for t in lst]
token_counts = Counter(all_tokens)

top_caps = token_counts.most_common(50)
top_caps_df = pd.DataFrame(top_caps, columns=['token', 'count'])
top_caps_df


,token,count
0,I,89943
1,GME,23635
2,AMC,7879
3,A,6420
4,THE,6250
5,DD,4928
6,WSB,4361
7,HOLD,3681
8,TO,3467
9,S,3427


In [38]:
REAL_TICKERS = {
    "GME",
    "AMC",
    "BB",
    "DD",
    "RKT",
    "NOK",
    "RH",
    "TSLA",
    "PLTR",
    "SPY",
    "IPO",
    "UWMC",
    "AMD",
    "SNDL",
    "CLOV",
    "SPCE",
    "AAPL",
    # you can add more later if needed
}
REAL_TICKERS


{'AAPL',
 'AMC',
 'AMD',
 'BB',
 'CLOV',
 'DD',
 'GME',
 'IPO',
 'NOK',
 'PLTR',
 'RH',
 'RKT',
 'SNDL',
 'SPCE',
 'SPY',
 'TSLA',
 'UWMC'}

In [39]:
def extract_base_tickers(text):
    text = str(text)
    up = text.upper()
    found = set()
    
    # 1) cashtags like $GME, $TSLA
    for tag in re.findall(r'\$[A-Z]{1,5}', up):
        sym = tag[1:]
        if sym in REAL_TICKERS:
            found.add(sym)
    
    # 2) standalone ticker symbols (e.g., " GME ", "TSLA," etc.)
    for sym in REAL_TICKERS:
        pattern = rf'\b{sym}\b'
        if re.search(pattern, up):
            found.add(sym)
    
    return list(found)

df['tickers_base'] = df['text'].apply(extract_base_tickers)
df['num_tickers_base'] = df['tickers_base'].apply(len)

df[['text', 'tickers_base']].head(15)


,text,tickers_base
0,"It's not about the money, it's about sending a...",[]
1,Math Professor Scott Steiner says the numbers ...,[]
2,Exit the system The CEO of NASDAQ pushed to ha...,[GME]
3,NEW SEC FILING FOR GME! CAN SOMEONE LESS RETAR...,[GME]
4,"Not to distract from GME, just thought our AMC...","[AMC, GME]"
5,WE BREAKING THROUGH,[]
6,SHORT STOCK DOESN'T HAVE AN EXPIRATION DATE He...,[GME]
7,THIS IS THE MOMENT Life isn't fair. My mother ...,"[BB, GME]"
8,Currently Holding AMC and NOK - Is it retarded...,"[AMC, NOK, GME]"
9,I have nothing to say but BRUH I am speechless...,[]


In [40]:
num_labeled_base = (df['num_tickers_base'] > 0).sum()
num_total = len(df)
pct_base = num_labeled_base / num_total * 100

num_labeled_base, num_total, round(pct_base, 2)


(22869, 53187, 43.0)

In [41]:
from collections import Counter

base_counts = Counter([t for lst in df['tickers_base'] for t in lst])
pd.DataFrame(base_counts.most_common(20), columns=['ticker', 'count'])


,ticker,count
0,GME,13584
1,AMC,5336
2,DD,3035
3,BB,2216
4,NOK,1695
5,RH,1418
6,PLTR,833
7,RKT,775
8,IPO,665
9,TSLA,602


In [42]:
NAME_TO_TICKER = {
    # GME
    "GAMESTOP": "GME",
    "GME": "GME",
    
    # AMC
    "AMC ENTERTAINMENT": "AMC",
    "AMC": "AMC",
    
    # BlackBerry
    "BLACKBERRY": "BB",
    "BB": "BB",
    
    # DuPont
    "DUPONT": "DD",
    "DD": "DD",
    
    # Rocket Companies
    "ROCKET": "RKT",
    "ROCKET MORTGAGE": "RKT",
    "ROCKET COMPANIES": "RKT",
    "RKT": "RKT",
    
    # Nokia
    "NOKIA": "NOK",
    "NOK": "NOK",
    
    # RH
    "RESTORATION HARDWARE": "RH",
    "RH": "RH",
    
    # Tesla
    "TESLA": "TSLA",
    "TSLA": "TSLA",
    
    # Palantir
    "PALANTIR": "PLTR",
    "PLTR": "PLTR",
    
    # S&P 500 ETF
    "S&P 500": "SPY",
    "S&P": "SPY",
    "SPY": "SPY",
    
    # IPO ETF
    "IPO": "IPO",
    
    # UWM Holdings
    "UWM": "UWMC",
    "UNITED WHOLESALE MORTGAGE": "UWMC",
    "UWMC": "UWMC",
    
    # AMD
    "ADVANCED MICRO DEVICES": "AMD",
    "AMD": "AMD",
    
    # Sundial
    "SUNDIAL": "SNDL",
    "SNDL": "SNDL",
    
    # Clover Health
    "CLOVER": "CLOV",
    "CLOV": "CLOV",
    
    # Virgin Galactic
    "VIRGIN GALACTIC": "SPCE",
    "SPCE": "SPCE",
    
    # Apple
    "APPLE": "AAPL",
    "AAPL": "AAPL",
}
len(NAME_TO_TICKER)


37

In [43]:
def add_name_based_tickers(row):
    text_up = str(row['text']).upper()
    tickers = set(row['tickers_base'])  # start from base tickers
    
    for name, ticker in NAME_TO_TICKER.items():
        if name in text_up:
            tickers.add(ticker)
    
    return list(tickers)

df['tickers'] = df.apply(add_name_based_tickers, axis=1)
df['num_tickers'] = df['tickers'].apply(len)

df[['text', 'tickers_base', 'tickers']].head(15)


,text,tickers_base,tickers
0,"It's not about the money, it's about sending a...",[],[]
1,Math Professor Scott Steiner says the numbers ...,[],[GME]
2,Exit the system The CEO of NASDAQ pushed to ha...,[GME],[GME]
3,NEW SEC FILING FOR GME! CAN SOMEONE LESS RETAR...,[GME],[GME]
4,"Not to distract from GME, just thought our AMC...","[AMC, GME]","[AMC, GME]"
5,WE BREAKING THROUGH,[],[]
6,SHORT STOCK DOESN'T HAVE AN EXPIRATION DATE He...,[GME],[GME]
7,THIS IS THE MOMENT Life isn't fair. My mother ...,"[BB, GME]","[RH, RKT, DD, BB, GME]"
8,Currently Holding AMC and NOK - Is it retarded...,"[AMC, NOK, GME]","[AMC, NOK, GME]"
9,I have nothing to say but BRUH I am speechless...,[],[]


In [44]:
num_labeled_final = (df['num_tickers'] > 0).sum()
pct_final = num_labeled_final / num_total * 100

(num_labeled_base, round(pct_base, 2)), (num_labeled_final, round(pct_final, 2))


((22869, 43.0), (28342, 53.29))

In [45]:
SUFFIX_VARIANTS = [
    "S",      
    "STOCK",   
    "STOCKS",  
    "SHARE",    
    "SHARES",   
]

def add_suffix_variants(row):
    text_up = str(row['text']).upper()
    tickers = set(row['tickers'])  # start from prior result

    for ticker in REAL_TICKERS:
        for suf in SUFFIX_VARIANTS:
            pattern = rf"\b{ticker}{suf}\b"
            if re.search(pattern, text_up):
                tickers.add(ticker)
    return list(tickers)

df['tickers'] = df.apply(add_suffix_variants, axis=1)
df['num_tickers'] = df['tickers'].apply(len)


In [46]:
SLANG_MAP = {
    # Tesla
    "ELON": "TSLA",
    "EV": "TSLA",

    # Gamestop
    "STONK": "GME",
    "TENDIES": "GME",
    "SQUEEZE": "GME",

    # AMC
    "MOVIE": "AMC",
    "THEATRE": "AMC",
    "POPCORN": "AMC",
}

def add_slang_tickers(row):
    text_up = str(row['text']).upper()
    tickers = set(row['tickers'])
    for word, ticker in SLANG_MAP.items():
        if word in text_up:
            tickers.add(ticker)
    return list(tickers)

df['tickers'] = df.apply(add_slang_tickers, axis=1)
df['num_tickers'] = df['tickers'].apply(len)


In [47]:
# compute for each date the % frequency of each ticker
exploded = df.explode('tickers')
daily_counts = exploded.groupby(['date', 'tickers']).size().reset_index(name='count')

# ticker that dominates each day
daily_dominant = daily_counts.sort_values(['date', 'count'], ascending=[True, False]) \
                             .groupby('date').first().reset_index()
daily_dominant.rename(columns={'tickers': 'dominant_ticker',
                               'count': 'dominant_count'}, inplace=True)

# merge back
df = df.merge(daily_dominant[['date', 'dominant_ticker', 'dominant_count']], on='date', how='left')

# assign ticker if none found but dominance ≥ 95%
def add_dominant_if_empty(row):
    if len(row['tickers']) == 0 and row['dominant_count'] / len(df[df['date'] == row['date']]) >= 0.95:
        return [row['dominant_ticker']]
    return row['tickers']

df['tickers'] = df.apply(add_dominant_if_empty, axis=1)
df['num_tickers'] = df['tickers'].apply(len)


In [48]:
num_labeled = (df['tickers'].apply(len) > 0).sum()
num_total = len(df)
pct = num_labeled / num_total * 100

num_labeled, num_total, round(pct, 2)


(33089, 53187, 62.21)

In [55]:
from collections import Counter
import pandas as pd

ticker_counts = Counter([t for lst in df['tickers'] for t in lst])
top50 = ticker_counts.most_common(50)
top50_df = pd.DataFrame(top50, columns=["ticker", "count"])
top50_df

,ticker,count
0,GME,17497
1,TSLA,16802
2,DD,9915
3,AMC,5695
4,BB,4390
5,RKT,2396
6,RH,2330
7,NOK,1945
8,PLTR,968
9,SPY,768


In [57]:
top10_auto = [t for t, _ in ticker_counts.most_common(10)]
TOP_TICKERS = top10_auto
top10_auto

['GME', 'TSLA', 'DD', 'AMC', 'BB', 'RKT', 'RH', 'NOK', 'PLTR', 'SPY']

In [58]:
def has_top_ticker(ticker_list):
    return any(t in TOP_TICKERS for t in ticker_list)

df_top = df[df['tickers'].apply(has_top_ticker)].copy()
df_top.shape

(32138, 10)

In [51]:
!pip install vaderSentiment


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 9.0 MB/s eta 0:00:00


In [65]:
# Hybrid sentiment analysis
# - Base: VADER compound sentiment score in [-1, 1]
# - Manual WSB-specific adjustments:
#       * Bullish keywords push score up
#       * Bearish keywords push score down
#   Final score is clipped to stay within [-1, 1]
#
# Outputs:
#   df_top['sentiment_score']  -> hybrid numeric score in [-1, 1]
#   df_top['sentiment_3']      -> "positive" / "neutral" / "negative."
#   df_top['sentiment_5']      -> "very positive" / "positive" / "neutral" / "negative" / "very negative."

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import numpy as np

# 1. Initialize VADER analyzer
analyzer = SentimentIntensityAnalyzer()

# 2. WSB-style bullish and bearish keywords 
BULLISH_WORDS = {
    "buy", "buying", "bought",
    "call", "calls",
    "moon", "mooning",
    "rocket", "rockets",
    "yolo",
    "hold", "holding", "hodl",
    "squeeze", "squeezing",
    "pump", "pumping",
    "bull", "bullish", "green."
}

BEARISH_WORDS = {
    "sell", "selling", "sold",
    "put", "puts",
    "tank", "tanking",
    "dump", "dumping",
    "bagholder", "bagholding",
    "paperhands",
    "bear", "bearish", "red",
    "crash", "crashing",
    "downbad"
}

def hybrid_sentiment(text: str) -> float:
    """
    Compute a hybrid sentiment score for a WSB post:
      1) Get VADER compound score in [-1, 1]
      2) Add +0.15 for each bullish keyword
      3) Subtract -0.15 for each bearish keyword
      4) normalize final result back into [-1, 1]
    """
    text_str = str(text)
    text_l = text_str.lower()
    
    # VADER score base
    vader_score = analyzer.polarity_scores(text_l)['compound']
    
    # WSB keyword boost
    boost = 0.0
    
    # Add for positive
    for w in BULLISH_WORDS:
        if w in text_l:
            boost += 0.15
    
    # Subtract for negative
    for w in BEARISH_WORDS:
        if w in text_l:
            boost -= 0.15
    
    # Combined score
    final_score = vader_score + boost
    final_score = max(-1.0, min(1.0, final_score))
    
    return final_score

# 3. Apply hybrid sentiment scoring to df_top
df_top['sentiment_score'] = df_top['text'].apply(hybrid_sentiment)

# 4. 3-class sentiment
def sentiment_label_3(score: float) -> str:
    """
      score >=  0.05  -> "positive"
      score <= -0.05  -> "negative"
      otherwise       -> "neutral."
    """
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

df_top['sentiment_3'] = df_top['sentiment_score'].apply(sentiment_label_3)

# 5. 5-class sentiment labels
def sentiment_label_5(score: float) -> str:
    """
      score >=  0.60           -> "very positive."
      0.05 <= score < 0.60     -> "positive"
     -0.05 < score < 0.05      -> "neutral"
     -0.60 <= score <= -0.05   -> "negative"
      score < -0.60            -> "very negative."
    """
    if score >= 0.60:
        return "very positive."
    elif 0.05 <= score < 0.60:
        return "positive"
    elif -0.05 < score < 0.05:
        return "neutral"
    elif -0.60 <= score <= -0.05:
        return "negative"
    else:
        return "very negative."

df_top['sentiment_5'] = df_top['sentiment_score'].apply(sentiment_label_5)

# 6. distribution of the 5-class labels
sent_5_dist = df_top['sentiment_5'].value_counts(normalize=True).rename("proportion")
sent_5_dist


sentiment_5
very positive.    0.384591
positive          0.241614
negative          0.166719
very negative.    0.104985
neutral           0.102091
Name: proportion, dtype: float64

In [66]:
df_top[['text', 'tickers', 'sentiment_score', 'sentiment_3', 'sentiment_5']].head(10)

,text,tickers,sentiment_score,sentiment_3,sentiment_5
1,Math Professor Scott Steiner says the numbers ...,[GME],-0.6249,negative,very negative.
2,Exit the system The CEO of NASDAQ pushed to ha...,[GME],-0.1644,negative,negative
3,NEW SEC FILING FOR GME! CAN SOMEONE LESS RETAR...,[GME],-0.3397,negative,negative
4,"Not to distract from GME, just thought our AMC...","[AMC, GME]",0.2235,positive,positive
6,SHORT STOCK DOESN'T HAVE AN EXPIRATION DATE He...,"[TSLA, GME]",0.9284,positive,very positive.
7,THIS IS THE MOMENT Life isn't fair. My mother ...,"[RH, RKT, DD, BB, GME, TSLA]",-0.8446,negative,very negative.
8,Currently Holding AMC and NOK - Is it retarded...,"[AMC, NOK, GME]",-0.2719,negative,negative
10,"We need to keep this movement going, we all ca...","[TSLA, AMC, GME]",1.0000,positive,very positive.
11,GME Premarket 🍁 Musk approved 🎮🛑💎✋,[GME],0.5859,positive,positive
12,"Once you're done with GME - $AG and $SLV, the ...","[DD, TSLA, GME, RKT]",1.0000,positive,very positive.


In [67]:
df_top['sentiment_score'].describe()

count    32138.000000
mean         0.284301
std          0.601885
min         -1.000000
25%         -0.150000
50%          0.340000
75%          0.859450
max          1.000000
Name: sentiment_score, dtype: float64

In [68]:
# Convert hybrid sentiment score from [-1, 1] to normalized [0, 1]
df_top['sentiment_norm'] = (df_top['sentiment_score'] + 1) / 2
df_top['sentiment_norm'].describe()


count    32138.000000
mean         0.642150
std          0.300943
min          0.000000
25%          0.425000
50%          0.670000
75%          0.929725
max          1.000000
Name: sentiment_norm, dtype: float64